In [ ]:
import random
import numpy as np
import time
import numba
import math
from numpy.random import Generator, PCG64DXSM, SeedSequence
import concurrent.futures
from structure_ORIGINAL_FNs_singlesideBASEreinLEARNING2_MODv2_2026 import play_sequence

np.set_printoptions(suppress=True)


In [ ]:
# use this cell for setting initial parameters

sides = 1
terms = 7
maxvalue = 10
startstop = 2
noise = 0.
annealing = 0.

timesteps = 10**8
runs = 1000

rein1 = 4
rein2 = 4
punish1 = -11
punish2 = -11
inertia = 1

nsteps = 100
blocklength = nsteps * 1
thrds = 3
nsteps_eval = 2100  # 50 plays per pair for 42 all-pairs


In [ ]:
def make_linear_pairs(n_stim):
    """
    Generate pair arrays for a linear transitive inference task with n_stim stimuli.
    
    n_stim should be odd. Adjacent pairs go in training. Non-adjacent pairs with
    at least one endpoint (0 or n_stim-1) go in training. Non-adjacent pairs
    where both elements are interior go to test pairs.
    
    Returns: (pairs, nonadjpairs, testpairs, pairs_reward, nonadjpairs_reward,
              testpairs_reward, allpairs, allpairs_reward)
    """
    pairs = []
    pairs_reward = []
    for i in range(n_stim - 1):
        for a, b in [(i, i + 1), (i + 1, i)]:
            pairs.append([a, b])
            pairs_reward.append([1, 0] if a > b else [0, 1])
    
    nonadjpairs = []
    nonadjpairs_reward = []
    testpairs = []
    testpairs_reward = []
    
    for i in range(n_stim):
        for j in range(n_stim):
            if i == j:
                continue
            if abs(i - j) <= 1:
                continue
            has_endpoint = (i == 0) or (i == n_stim - 1) or (j == 0) or (j == n_stim - 1)
            reward = [1, 0] if i > j else [0, 1]
            if has_endpoint:
                nonadjpairs.append([i, j])
                nonadjpairs_reward.append(reward)
            else:
                testpairs.append([i, j])
                testpairs_reward.append(reward)
    
    allpairs = pairs + nonadjpairs + testpairs
    allpairs_reward = pairs_reward + nonadjpairs_reward + testpairs_reward
    
    return (np.asarray(pairs, dtype=np.int64),
            np.asarray(nonadjpairs, dtype=np.int64),
            np.asarray(testpairs, dtype=np.int64),
            np.asarray(pairs_reward, dtype=np.int64),
            np.asarray(nonadjpairs_reward, dtype=np.int64),
            np.asarray(testpairs_reward, dtype=np.int64),
            np.asarray(allpairs, dtype=np.int64),
            np.asarray(allpairs_reward, dtype=np.int64))


# ---- circular reward variant ----
# Uncomment to use circular ordering instead of linear ordering.
# For pair (x, y) with y > x: d_x = y - x, d_y = (x - y) % n_stim.
# Reward the element that is "rightmost" in the shortest circular path.
# E.g. terms=7: pair (0,3) -> path 0,1,2,3 -> reward 3; pair (0,4) -> path 4,5,6,0 -> reward 0.

# def make_circular_pairs(n_stim):
#     """
#     Generate pair arrays for a circular transitive inference task with n_stim stimuli.
#     
#     n_stim should be odd. Adjacent pairs on the circle (including the wrap-around pair)
#     go to training. Non-adjacent pairs with at least one endpoint (0 or n_stim-1)
#     go to training. Non-adjacent interior pairs go to test.
#     
#     Rewards follow the circular shortest-path rule: for pair (x, y) with y > x,
#     d_x = y - x, d_y = (x - y) % n_stim. Reward y if d_x < d_y, reward x if d_x > d_y.
#     
#     Returns: (pairs, nonadjpairs, testpairs, pairs_reward, nonadjpairs_reward,
#               testpairs_reward, allpairs, allpairs_reward)
#     """
#     pairs = []
#     pairs_reward = []
#     # Adjacent pairs on the circle: (i, i+1) for i in 0..n_stim-2, plus (0, n_stim-1) wrap-around
#     for i in range(n_stim - 1):
#         for a, b in [(i, i + 1), (i + 1, i)]:
#             pairs.append([a, b])
#             x, y = min(a, b), max(a, b)
#             dx = y - x
#             dy = (x - y) % n_stim
#             pairs_reward.append([1, 0] if a == (y if dx < dy else x) else [0, 1])
#     # Wrap-around adjacent: (0, n_stim-1) and (n_stim-1, 0)
#     for a, b in [(0, n_stim - 1), (n_stim - 1, 0)]:
#         pairs.append([a, b])
#         x, y = min(a, b), max(a, b)
#         dx = y - x
#         dy = (x - y) % n_stim
#         pairs_reward.append([1, 0] if a == (y if dx < dy else x) else [0, 1])
#     
#     nonadjpairs = []
#     nonadjpairs_reward = []
#     testpairs = []
#     testpairs_reward = []
#     
#     for i in range(n_stim):
#         for j in range(n_stim):
#             if i == j:
#                 continue
#             # Skip linearly adjacent pairs (already in pairs)
#             if abs(i - j) == 1:
#                 continue
#             # Skip wrap-around adjacent (0, n_stim-1) and (n_stim-1, 0)
#             if set([i, j]) == set([0, n_stim - 1]):
#                 continue
#             x, y = min(i, j), max(i, j)
#             dx = y - x
#             dy = (x - y) % n_stim
#             rewarded = y if dx < dy else x
#             reward = [1, 0] if i == rewarded else [0, 1]
#             has_endpoint = (i == 0) or (i == n_stim - 1) or (j == 0) or (j == n_stim - 1)
#             if has_endpoint:
#                 nonadjpairs.append([i, j])
#                 nonadjpairs_reward.append(reward)
#             else:
#                 testpairs.append([i, j])
#                 testpairs_reward.append(reward)
#     
#     allpairs = pairs + nonadjpairs + testpairs
#     allpairs_reward = pairs_reward + nonadjpairs_reward + testpairs_reward
#     
#     return (np.asarray(pairs, dtype=np.int64),
#             np.asarray(nonadjpairs, dtype=np.int64),
#             np.asarray(testpairs, dtype=np.int64),
#             np.asarray(pairs_reward, dtype=np.int64),
#             np.asarray(nonadjpairs_reward, dtype=np.int64),
#             np.asarray(testpairs_reward, dtype=np.int64),
#             np.asarray(allpairs, dtype=np.int64),
#             np.asarray(allpairs_reward, dtype=np.int64))


pairs, nonadjpairs, testpairs, pairs_reward, nonadjpairs_reward, testpairs_reward, allpairs, allpairs_reward = make_linear_pairs(terms)
plen = len(pairs)
alen = len(allpairs)


In [ ]:
# use this cell for running the code

start = time.perf_counter()

# Seed sequence: print entropy so results can be reproduced without saved arrays
sq1 = SeedSequence()
randomentropy = sq1.entropy
print(f'Initial seed entropy: {randomentropy}')
sg = SeedSequence(randomentropy)
rgs = numba.typed.List([Generator(PCG64DXSM(s)) for s in sg.spawn(runs)])

# Run all processes
with concurrent.futures.ProcessPoolExecutor(max_workers=thrds) as executor:
    future_to_run = {
        executor.submit(
            play_sequence,
            r, rgs[r], rein1, punish1, rein2, punish2,
            timesteps, nsteps, sides,
            pairs, testpairs, nonadjpairs, allpairs,
            pairs_reward, testpairs_reward, nonadjpairs_reward, allpairs_reward,
            plen, alen, terms, maxvalue, startstop,
            noise, annealing, runs, inertia, blocklength, nsteps_eval
        ): r for r in range(runs)
    }
    # Collect results into pre-allocated arrays keyed by run ID
    mpseq_final = [None] * runs
    for future in concurrent.futures.as_completed(future_to_run):
        data_runid = future_to_run[future]
        try:
            result = future.result()
        except Exception as exc:
            print(f'Run #{data_runid} generated an exception: {exc}')
        else:
            mpseq_final[data_runid] = result
            if data_runid % 100 == 0:
                print(f'finished run #{data_runid}')

# Unpack results
final_sigweights = np.asarray([r[0] for r in mpseq_final])
final_cumsuc = np.asarray([r[1] for r in mpseq_final])
final_cumsucadd = np.asarray([r[2] for r in mpseq_final])
final_testcumsucadd = np.asarray([r[3] for r in mpseq_final])
final_recweights = np.asarray([r[4] for r in mpseq_final])
final_test_pair_stats = np.asarray([r[5] for r in mpseq_final])
final_evalcumsucadd = np.asarray([r[6] for r in mpseq_final])
final_eval_pair_stats = np.asarray([r[7] for r in mpseq_final])

print(f'average cumsuc = {np.average(final_cumsuc) / runs}')
print(f'average final cumsucadd = {np.average(final_cumsucadd) / nsteps}')
print(f'average test cumsum = {np.average(final_testcumsucadd) / nsteps_eval}')
print(f'average all-pairs eval = {np.average(final_evalcumsucadd) / nsteps_eval}')

finish = time.perf_counter()
print(f'Finished in {round(finish - start, 0) / 60} minutes')


In [ ]:
# Save results
np.save('structure_noiseAnn_dsktp_1s_BASElearning-MV10_sigweights', final_sigweights)
np.save('structure_noiseAnn_dsktp_1s_BASElearning-MV10_recweights', final_recweights)
np.save('structure_noiseAnn_dsktp_1s_BASElearning-MV10_testcumsucadd', final_testcumsucadd)
np.save('structure_noiseAnn_dsktp_1s_BASElearning-MV10_test_pair_stats', final_test_pair_stats)
np.save('structure_noiseAnn_dsktp_1s_BASElearning-MV10_evalcumsucadd', final_evalcumsucadd)
np.save('structure_noiseAnn_dsktp_1s_BASElearning-MV10_eval_pair_stats', final_eval_pair_stats)


In [ ]:
# Per-pair accuracy on test pairs
print('Test pair stats (attempts, successes per pair):')
for i, pair in enumerate(testpairs):
    attempts = final_test_pair_stats[:, i, 0].sum()
    successes = final_test_pair_stats[:, i, 1].sum()
    acc = successes / attempts if attempts > 0 else 0
    dist = abs(pair[0] - pair[1])
    print(f'  Pair {pair}: accuracy = {acc:.4f} ({successes}/{attempts}), distance = {dist}')


In [ ]:
# Symbolic distance effect: aggregate test-pair accuracy by distance
print('Test accuracy by distance:')
distances_test = np.abs(testpairs[:, 0] - testpairs[:, 1])
unique_dists = np.unique(distances_test)
for d in unique_dists:
    mask = distances_test == d
    idxs = np.where(mask)[0]
    total_attempts = 0
    total_successes = 0
    for idx in idxs:
        total_attempts += final_test_pair_stats[:, idx, 0].sum()
        total_successes += final_test_pair_stats[:, idx, 1].sum()
    acc = total_successes / total_attempts if total_attempts > 0 else 0
    print(f'  Distance {d}: accuracy = {acc:.4f} ({total_successes}/{total_attempts})')


In [ ]:
# All-pairs eval accuracy by distance
print('All-pairs eval accuracy by distance:')
distances_eval = np.abs(allpairs[:, 0] - allpairs[:, 1])
unique_dists_eval = np.unique(distances_eval)
for d in unique_dists_eval:
    mask = distances_eval == d
    idxs = np.where(mask)[0]
    total_attempts = 0
    total_successes = 0
    for idx in idxs:
        total_attempts += final_eval_pair_stats[:, idx, 0].sum()
        total_successes += final_eval_pair_stats[:, idx, 1].sum()
    acc = total_successes / total_attempts if total_attempts > 0 else 0
    print(f'  Distance {d}: accuracy = {acc:.4f} ({total_successes}/{total_attempts})')


In [ ]:
# Serial position effect: aggregate test-pair accuracy by stimulus endpoints
print('Test accuracy by endpoint positions:')
for i, pair in enumerate(testpairs):
    attempts = final_test_pair_stats[:, i, 0].sum()
    successes = final_test_pair_stats[:, i, 1].sum()
    acc = successes / attempts if attempts > 0 else 0
    print(f'  Pair [{pair[0]}, {pair[1]}]: endpoints=({pair[0]}, {pair[1]}), accuracy = {acc:.4f} ({successes}/{attempts})')


In [ ]:
# All-pairs eval accuracy by endpoint positions (serial position effect)
print('All-pairs eval accuracy by endpoint positions:')
for i, pair in enumerate(allpairs):
    attempts = final_eval_pair_stats[:, i, 0].sum()
    successes = final_eval_pair_stats[:, i, 1].sum()
    acc = successes / attempts if attempts > 0 else 0
    dist = abs(pair[0] - pair[1])
    adj_label = 'adj' if dist == 1 else f'dist-{dist}'
    print(f'  Pair [{pair[0]}, {pair[1]}]: {adj_label}, accuracy = {acc:.4f} ({successes}/{attempts})')


In [ ]:
# Count runs exceeding cutoff
cutoff = int(0.9 * nsteps_eval)  # 90% threshold
final_test_count = int((final_testcumsucadd > cutoff).sum())
print(f'Runs with test score > {cutoff}: {final_test_count}')


In [ ]:
@numba.njit
def duplicates(dbins):
    dup = 0
    for iii in range(len(dbins)):
        for jjj in range(len(dbins)):
            if iii != jjj:
                if (dbins[iii][0] == dbins[jjj][0]) & (dbins[iii][1] == dbins[jjj][1]):
                    dup = 1
    return dup

# function to check how many runs had duplicate bins
@numba.njit
def runs_dup_bins(runs, fsigweights, t):
    dup_count = 0
    for i0 in range(runs):
        sw = fsigweights[i0].copy()
        swl = sw.argmax(axis=3)[0, 0]
        swr = sw.argmax(axis=3)[0, 1]
        bns = np.zeros(((2*(t-1)), 2), dtype=numba.int64)
        for idx000 in range(t-1):
            bns[idx000][0] = swl[idx000]
            bns[idx000][1] = swr[idx000+1]
        for idx001 in range(t-1):
            bns[t-1+idx001][0] = swl[idx001+1]
            bns[t-1+idx001][1] = swr[idx001]
        dup_count += duplicates(bns)
    return dup_count


In [ ]:
runs_dup_bins(runs, final_sigweights, terms)
